# 09 — Interpretability Figures and Tables

This notebook prepares the **Chapter 7 thesis outputs** for interpretability and physiology-oriented discussion.

## Main purpose

This notebook turns the outputs of the earlier interpretability analyses into **thesis-ready figures and tables**.

It focuses on the **strongest classical baseline (LDA)** and prepares evidence for statements such as:

- the strongest classical baseline uses **distributed post-stimulus spatiotemporal structure**
- the decoded signal is **not dominated globally by gamma-band activity**
- **theta contributes most strongly overall**
- some **peripheral gamma structure remains**, so muscle-related contributions cannot yet be ruled out completely

## Main outputs

This notebook will produce:

- **Figure 7.1** — Broadband interpretability of the strongest classical baseline
- **Figure 7.2** — Bandpower diagnostic figure
- **Table 7.1** — Broadband interpretability helper summary
- **Table 7.2** — Bandpower diagnostic helper summary

## Important note

This notebook is for **figure and table preparation**, not for rerunning the benchmark.
It assumes the earlier interpretability outputs already exist.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from oddball_eeg.config import get_experiment_family, SIGNAL_0P1_45
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_json
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)

## 1. Setup paths and notebook options

This notebook tries to find the required interpretability CSV files automatically.

It looks first in the current directory, then in common subdirectories such as:

- `interpretability/`
- `bandpower_importance/`
- `results/interpretability/`
- `results/bandpower_importance/`

You can edit the search roots below if needed.

In [ ]:
set_pandas_display_options()

# Change only this line when switching datasets.
EXPERIMENT_NAME = "ds005863_within_subject_main"
# EXPERIMENT_NAME = "ds003061_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)

# Broadband interpretability uses the main benchmark signal, normally 0.1–12 Hz.
cfg_main = experiment_family["main"]

# Bandpower diagnostic uses a 0.1–45 Hz diagnostic signal so gamma is available.
cfg_bandpower = replace(
    cfg_main,
    name="bandpower_diagnostic",
    signal=SIGNAL_0P1_45,
)

BROADBAND_TABLE_DIR = cfg_main.table_dir("interpretability", "broadband")
BANDPOWER_TABLE_DIR = cfg_bandpower.table_dir("interpretability", "bandpower")

OUT_DIR = cfg_main.artifact_dir("chapter7_interpretability")
FIG_DIR = cfg_main.artifact_dir("chapter7_interpretability", "figures")
TABLE_DIR = cfg_main.artifact_dir("chapter7_interpretability", "tables")

PRIMARY_MODEL = "lda"
P300_TMIN = 0.25
P300_TMAX = 0.50

save_experiment_metadata(
    cfg_main,
    extra={
        "notebook": "09_interpretability_figures_tables.ipynb",
        "purpose": "Generate Chapter 7 interpretability figures and tables from saved broadband and bandpower outputs.",
        "experiment_key": EXPERIMENT_NAME,
        "dataset": cfg_main.dataset.dataset,
        "task": cfg_main.dataset.task,
        "primary_model": PRIMARY_MODEL,
        "p300_window": [P300_TMIN, P300_TMAX],
        "broadband_table_dir": str(BROADBAND_TABLE_DIR),
        "bandpower_table_dir": str(BANDPOWER_TABLE_DIR),
        "artifact_dir": str(OUT_DIR),
        "figure_dir": str(FIG_DIR),
        "table_dir": str(TABLE_DIR),
    },
)

print("Experiment key     :", EXPERIMENT_NAME)
print("Dataset            :", cfg_main.dataset.dataset)
print("Task               :", cfg_main.dataset.task)
print("Broadband signal   :", cfg_main.signal.slug())
print("Bandpower signal   :", cfg_bandpower.signal.slug())
print("Primary model      :", PRIMARY_MODEL)
print("P300/ERP window    :", (P300_TMIN, P300_TMAX))

print("\nInput directories:")
print("Broadband tables   :", BROADBAND_TABLE_DIR)
print("Bandpower tables   :", BANDPOWER_TABLE_DIR)

print("\nOutput directories:")
print("Artifacts          :", OUT_DIR)
print("Figures            :", FIG_DIR)
print("Tables             :", TABLE_DIR)

In [ ]:
def read_required_csv(path: Path) -> pd.DataFrame:
    """
    Read a required CSV file and raise a clear error if it is missing.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{path}\n\n"
            "Make sure the upstream notebook that creates this file has been run."
        )

    return pd.read_csv(path)

## 2. Load broadband interpretability outputs

We now load the main outputs from the earlier coefficient-based interpretability analysis.

Expected files:

- `mean_time_importance.csv`
- `mean_electrode_importance.csv`
- `subject_time_importance.csv`
- `subject_electrode_importance.csv`

If your filenames differ slightly, just update the names below.

In [ ]:
time_mean_path = BROADBAND_TABLE_DIR / "mean_time_importance.csv"
elec_mean_path = BROADBAND_TABLE_DIR / "mean_electrode_importance.csv"
time_subject_path = BROADBAND_TABLE_DIR / "subject_time_importance.csv"
elec_subject_path = BROADBAND_TABLE_DIR / "subject_electrode_importance.csv"

df_time_mean = read_required_csv(time_mean_path)
df_elec_mean = read_required_csv(elec_mean_path)
df_time_subject = read_required_csv(time_subject_path)
df_elec_subject = read_required_csv(elec_subject_path)

print("Loaded broadband interpretability files:")
print("-", time_mean_path)
print("-", elec_mean_path)
print("-", time_subject_path)
print("-", elec_subject_path)

In [ ]:
display(round_numeric_columns(df_time_mean.head().copy(), 4))
display(round_numeric_columns(df_elec_mean.head().copy(), 4))

## 3. Load bandpower diagnostic outputs

We now load the outputs from the bandpower diagnostic analysis.

Expected files:

- `mean_band_importance.csv`
- `mean_band_time_importance.csv`
- `mean_band_electrode_importance.csv`
- `lda_band_importance_summary.csv`
- `lda_gamma_top_electrodes.csv`

In [ ]:
# Load bandpower diagnostic outputs from Notebook 07

band_mean_path = BANDPOWER_TABLE_DIR / "mean_band_importance.csv"
band_time_mean_path = BANDPOWER_TABLE_DIR / "mean_band_time_importance.csv"
band_elec_mean_path = BANDPOWER_TABLE_DIR / "mean_band_electrode_importance.csv"

df_band_mean = read_required_csv(band_mean_path)
df_band_time_mean = read_required_csv(band_time_mean_path)
df_band_elec_mean = read_required_csv(band_elec_mean_path)

print("Loaded bandpower files:")
print("-", band_mean_path)
print("-", band_time_mean_path)
print("-", band_elec_mean_path)

display(round_numeric_columns(df_band_mean.copy(), 4))
display(round_numeric_columns(df_band_elec_mean.head(10).copy(), 4))

In [ ]:
MODEL_NAME_MAP = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}


def normalize_model_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model" in out.columns:
        out["model"] = out["model"].astype(str).str.strip().str.lower()
    return out


def get_electrode_importance_cols(df: pd.DataFrame):
    candidates = [
        ("importance_p300_mean", "importance_p300_std"),
        ("p300_importance_mean", "p300_importance_std"),
        ("importance_mean_p300", "importance_std_p300"),
        ("importance_mean", "importance_std"),
    ]

    for mean_col, std_col in candidates:
        if mean_col in df.columns:
            return mean_col, std_col if std_col in df.columns else None

    raise KeyError(
        "Could not find an electrode importance mean column. "
        f"Available columns: {df.columns.tolist()}"
    )


def filter_primary_model(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["model"] == PRIMARY_MODEL].copy()


df_time_mean = normalize_model_column(df_time_mean)
df_elec_mean = normalize_model_column(df_elec_mean)
df_time_subject = normalize_model_column(df_time_subject)
df_elec_subject = normalize_model_column(df_elec_subject)

df_band_mean = normalize_model_column(df_band_mean)
df_band_time_mean = normalize_model_column(df_band_time_mean)
df_band_elec_mean = normalize_model_column(df_band_elec_mean)

# Broadband time importance
time_lda = (
    filter_primary_model(df_time_mean)
    .groupby("time_s", as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values("time_s")
    .reset_index(drop=True)
)

# Broadband electrode importance
elec_raw = filter_primary_model(df_elec_mean).copy()
elec_mean_col, elec_std_col = get_electrode_importance_cols(elec_raw)

agg_kwargs = {
    elec_mean_col: (elec_mean_col, "mean"),
}

if elec_std_col is not None:
    agg_kwargs[elec_std_col] = (elec_std_col, "mean")

elec_lda = (
    elec_raw
    .groupby("channel", as_index=False)
    .agg(**agg_kwargs)
    .sort_values(elec_mean_col, ascending=False)
    .reset_index(drop=True)
)

# Band summary
band_lda = (
    filter_primary_model(df_band_mean)
    .groupby("band", as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

# Band time importance
band_time_lda = (
    filter_primary_model(df_band_time_mean)
    .groupby(["band", "time_s"], as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values(["band", "time_s"])
    .reset_index(drop=True)
)

# Band electrode importance
band_elec_lda = (
    filter_primary_model(df_band_elec_mean)
    .groupby(["band", "channel"], as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values(["band", "importance_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

ch_names = elec_lda["channel"].astype(str).drop_duplicates().tolist()

if len(time_lda) == 0:
    raise ValueError(f"No broadband time-importance rows found for model={PRIMARY_MODEL}")

if len(elec_lda) == 0:
    raise ValueError(f"No broadband electrode-importance rows found for model={PRIMARY_MODEL}")

if len(band_lda) == 0:
    raise ValueError(f"No bandpower rows found for model={PRIMARY_MODEL}")

print("Primary model :", PRIMARY_MODEL)
print("n time points :", len(time_lda))
print("n channels    :", len(ch_names))
print("bands         :", sorted(band_lda["band"].unique().tolist()))

In [ ]:
display(round_numeric_columns(df_band_mean.copy(), 4))
display(round_numeric_columns(df_band_elec_mean.head(10).copy(), 4))

## 4. Infer shared metadata

For the broadband figures, we need:

- the LDA time axis
- the channel list
- basic model filtering

This notebook uses the aggregated outputs directly.

In [ ]:
MODEL_NAME_MAP = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}

def normalize_model_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model" in out.columns:
        out["model"] = out["model"].astype(str).str.strip().str.lower()
    return out

def find_preproc_col(*dfs) -> str | None:
    candidates = ["preprocessing", "preproc", "condition", "pipeline", "input_type", "feature_set"]
    for df in dfs:
        for c in candidates:
            if c in df.columns:
                return c
    return None

def choose_primary_preproc(labels: list[str]) -> str | None:
    normalized = {lab: str(lab).strip().lower().replace("–", "-") for lab in labels}
    compact = {lab: norm.replace(" ", "").replace("hz", "") for lab, norm in normalized.items()}

    priority_patterns = [
        "0.1-12",
        "0.1to12",
        "0.1_12",
        "bp_0.1_12",
        "bandpass_0.1_12",
        "lowfreq",
        "erp",
    ]

    for pat in priority_patterns:
        for raw, norm in compact.items():
            if pat in norm:
                return raw
    return None

def get_electrode_importance_cols(df: pd.DataFrame):
    candidates = [
        ("importance_p300_mean", "importance_p300_std"),
        ("p300_importance_mean", "p300_importance_std"),
        ("importance_mean_p300", "importance_std_p300"),
        ("importance_mean", "importance_std"),
    ]
    for mean_col, std_col in candidates:
        if mean_col in df.columns:
            return mean_col, std_col if std_col in df.columns else None
    raise KeyError(
        "Could not find an electrode importance mean column. "
        f"Available columns: {df.columns.tolist()}"
    )

df_time_mean = normalize_model_column(df_time_mean)
df_elec_mean = normalize_model_column(df_elec_mean)
df_time_subject = normalize_model_column(df_time_subject)
df_elec_subject = normalize_model_column(df_elec_subject)

df_band_mean = normalize_model_column(df_band_mean)
df_band_time_mean = normalize_model_column(df_band_time_mean)
df_band_elec_mean = normalize_model_column(df_band_elec_mean)

PREPROC_COL = find_preproc_col(
    df_time_mean, df_elec_mean, df_band_mean, df_band_time_mean, df_band_elec_mean
)

PRIMARY_PREPROC = None
if PREPROC_COL is not None:
    for _df in [df_time_mean, df_elec_mean, df_time_subject, df_elec_subject,
                df_band_mean, df_band_time_mean, df_band_elec_mean]:
        if PREPROC_COL in _df.columns:
            _df[PREPROC_COL] = _df[PREPROC_COL].astype(str).str.strip()

    available_preprocs = sorted(
        pd.concat([
            df_time_mean[PREPROC_COL] if PREPROC_COL in df_time_mean.columns else pd.Series(dtype=str),
            df_band_time_mean[PREPROC_COL] if PREPROC_COL in df_band_time_mean.columns else pd.Series(dtype=str),
        ]).dropna().astype(str).unique().tolist()
    )

    PRIMARY_PREPROC = choose_primary_preproc(available_preprocs)

    print("Detected preprocessing column:", PREPROC_COL)
    print("Available preprocessing labels:", available_preprocs)
    print("Selected preprocessing label :", PRIMARY_PREPROC)

    if PRIMARY_PREPROC is None:
        raise ValueError(
            "Could not auto-detect the final 0.1–12 Hz pipeline label. "
            "Please set PRIMARY_PREPROC manually in this cell."
        )
else:
    print("No preprocessing column detected; proceeding without preprocessing filter.")

def filter_final_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    out = df[df["model"] == PRIMARY_MODEL].copy()
    if PREPROC_COL is not None and PREPROC_COL in out.columns:
        out = out[out[PREPROC_COL] == PRIMARY_PREPROC].copy()
    return out

# Broadband time importance
time_lda = (
    filter_final_pipeline(df_time_mean)
    .groupby("time_s", as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values("time_s")
    .reset_index(drop=True)
)

# Broadband electrode importance
elec_raw = filter_final_pipeline(df_elec_mean).copy()
elec_mean_col, elec_std_col = get_electrode_importance_cols(elec_raw)

agg_kwargs = {
    elec_mean_col: (elec_mean_col, "mean"),
}
if elec_std_col is not None:
    agg_kwargs[elec_std_col] = (elec_std_col, "mean")

elec_lda = (
    elec_raw.groupby("channel", as_index=False)
    .agg(**agg_kwargs)
    .sort_values(elec_mean_col, ascending=False)
    .reset_index(drop=True)
)

# Band summary
band_lda = (
    filter_final_pipeline(df_band_mean)
    .groupby("band", as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

# Band time importance
band_time_lda = (
    filter_final_pipeline(df_band_time_mean)
    .groupby(["band", "time_s"], as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values(["band", "time_s"])
    .reset_index(drop=True)
)

# Band electrode importance
band_elec_lda = (
    filter_final_pipeline(df_band_elec_mean)
    .groupby(["band", "channel"], as_index=False)
    .agg(
        importance_mean=("importance_mean", "mean"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values(["band", "importance_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

ch_names = elec_lda["channel"].astype(str).drop_duplicates().tolist()

print("Primary model :", PRIMARY_MODEL)
print("Primary preproc:", PRIMARY_PREPROC)
print("n time points :", len(time_lda))
print("n channels    :", len(ch_names))

## 5. Create helper functions

We now define plotting and table-export helpers for:

- broadband time importance
- broadband electrode ranking
- broadband topomap
- bandpower summaries
- LaTeX export

In [ ]:
def save_latex_table(
        df: pd.DataFrame,
        path: Path,
        index: bool = False,
        float_format: str = "%.4f",
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    latex = df.to_latex(index=index, escape=False, float_format=float_format)
    path.write_text(latex, encoding="utf-8")


def build_mne_info(ch_names: list[str], sfreq: float = 128.0) -> mne.Info:
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    info.set_montage("standard_1020", on_missing="ignore")
    return info

## 6. Figure 7.1 — Broadband interpretability of the strongest classical baseline

This figure combines:

- **left panel:** mean time-point importance
- **right panel:** P300-window electrode importance

If valid channel positions are available, the right panel is drawn as a topomap.
Otherwise, the notebook falls back to a ranked bar chart.

In [ ]:
def plot_broadband_two_panel(
        time_lda: pd.DataFrame,
        elec_lda: pd.DataFrame,
        p300_tmin: float,
        p300_tmax: float,
        top_n: int = 10,
):
    mean_col, std_col = get_electrode_importance_cols(elec_lda)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left panel: time importance
    ax = axes[0]
    ax.plot(time_lda["time_s"], time_lda["importance_mean"], linewidth=2)
    ax.axvline(0.0, linestyle="--", alpha=0.7)
    ax.axvspan(p300_tmin, p300_tmax, alpha=0.15)
    ax.set_title(
        f"{MODEL_NAME_MAP.get(PRIMARY_MODEL, PRIMARY_MODEL.upper())} broadband time-point importance"
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mean normalized |coefficient|")
    ax.grid(True, alpha=0.3)

    # Right panel: ranked electrodes
    ax = axes[1]
    top_df = (
        elec_lda[["channel", mean_col]]
        .sort_values(mean_col, ascending=False)
        .head(top_n)
        .sort_values(mean_col, ascending=True)
    )
    ax.barh(top_df["channel"], top_df[mean_col])
    ax.set_title(
        f"{MODEL_NAME_MAP.get(PRIMARY_MODEL, PRIMARY_MODEL.upper())} top electrodes"
    )
    ax.set_xlabel("Mean normalized importance")
    ax.grid(True, axis="x", alpha=0.3)

    fig.suptitle(
        f"Broadband interpretability — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
        y=1.03,
    )
    fig.tight_layout()
    return fig

In [ ]:
fig_7_1 = plot_broadband_two_panel(
    time_lda=time_lda,
    elec_lda=elec_lda,
    p300_tmin=P300_TMIN,
    p300_tmax=P300_TMAX,
)

save_figure(
    fig_7_1,
    FIG_DIR / f"figure_7_1_{PRIMARY_MODEL}_broadband_interpretability.png",
)
plt.show()

## 7. Build broadband helper tables

These tables help support the Chapter 7 writing.

We prepare:

- top electrodes in the P300 window
- strongest time points
- simple window summaries

In [ ]:
# Detect the correct electrode-importance columns
electrode_mean_col = None
electrode_std_col = None

candidate_pairs = [
    ("importance_p300_mean", "importance_p300_std"),
    ("p300_importance_mean", "p300_importance_std"),
    ("importance_mean_p300", "importance_std_p300"),
    ("importance_mean", "importance_std"),
]

for mean_col, std_col in candidate_pairs:
    if mean_col in elec_lda.columns:
        electrode_mean_col = mean_col
        electrode_std_col = std_col if std_col in elec_lda.columns else None
        break

if electrode_mean_col is None:
    raise KeyError(
        f"Could not find electrode importance columns in elec_lda. "
        f"Available columns: {elec_lda.columns.tolist()}"
    )

# Build electrode table
electrode_cols = ["channel", electrode_mean_col]
if electrode_std_col is not None:
    electrode_cols.append(electrode_std_col)

top_electrodes_p300 = (
    elec_lda[electrode_cols]
    .sort_values(electrode_mean_col, ascending=False)
    .reset_index(drop=True)
    .rename(columns={
        electrode_mean_col: "importance_mean",
        electrode_std_col: "importance_std" if electrode_std_col is not None else electrode_mean_col,
    })
)

# Build time-point table
top_time_points = (
    time_lda[["time_s", "importance_mean", "importance_std"]]
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

# Window summary
time_windows = [
    ("early", float(time_lda["time_s"].min()), 0.15),
    ("middle", 0.15, 0.35),
    ("late", 0.35, float(time_lda["time_s"].max())),
]

rows = []
for label, tmin, tmax in time_windows:
    mask = (time_lda["time_s"] >= tmin) & (time_lda["time_s"] <= tmax)
    rows.append({
        "window": label,
        "tmin_s": tmin,
        "tmax_s": tmax,
        "mean_importance": time_lda.loc[mask, "importance_mean"].mean(),
    })

window_summary = pd.DataFrame(rows)

display(round_numeric_columns(top_electrodes_p300.head(12).copy(), 4))
display(round_numeric_columns(top_time_points.head(12).copy(), 4))
display(round_numeric_columns(window_summary.copy(), 4))

In [ ]:
save_dataframe(
    top_electrodes_p300,
    TABLE_DIR / "table_7_1a_top_electrodes_p300.csv",
)

save_dataframe(
    top_time_points,
    TABLE_DIR / "table_7_1b_top_time_points.csv",
)

save_dataframe(
    window_summary,
    TABLE_DIR / "table_7_1c_time_window_summary.csv",
)

save_latex_table(
    round_numeric_columns(top_electrodes_p300.head(12).copy(), 4),
    TABLE_DIR / "table_7_1a_top_electrodes_p300.tex",
    index=False,
)
save_latex_table(
    round_numeric_columns(top_time_points.head(12).copy(), 4),
    TABLE_DIR / "table_7_1b_top_time_points.tex",
    index=False,
)
save_latex_table(
    round_numeric_columns(window_summary.copy(), 4),
    TABLE_DIR / "table_7_1c_time_window_summary.tex",
    index=False,
)

print("Saved broadband helper tables.")

## 8. Figure 7.2 — Bandpower diagnostic

This figure should support the Chapter 7 claim that:

- theta contributes most strongly overall
- gamma is not globally dominant
- some peripheral gamma structure remains

We will create a compact **2 × 3 figure**:

- top row: time importance for theta / alpha / gamma
- bottom row: topographic importance for theta / alpha / gamma

In [ ]:
def plot_bandpower_report_figure(
        band_time_lda: pd.DataFrame,
        band_elec_lda: pd.DataFrame,
        ch_names: list[str],
        late_tmin: float,
        late_tmax: float,
        sfreq_for_topomap: float = 128.0,
):
    band_order = ["theta", "alpha", "gamma"]

    def unique_preserve_order(seq):
        seen = set()
        out = []
        for x in seq:
            x = str(x).strip()
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    def make_topomap_info(channel_names, sfreq):
        info = mne.create_info(ch_names=channel_names, sfreq=sfreq, ch_types="eeg")
        montage = mne.channels.make_standard_montage("standard_1020")
        info.set_montage(montage, on_missing="ignore")
        return info

    def get_valid_topomap_picks(info):
        picks = []
        for idx, ch in enumerate(info["chs"]):
            loc = ch["loc"][:3]
            if np.isfinite(loc).all() and not np.allclose(loc, 0):
                picks.append(idx)
        return picks

    ch_names_unique = unique_preserve_order(ch_names)

    info_valid = None
    valid_ch_names = None
    try:
        info = make_topomap_info(ch_names_unique, sfreq_for_topomap)
        picks = get_valid_topomap_picks(info)
        if len(picks) >= 4:
            info_valid = mne.pick_info(info, picks, copy=True)
            valid_ch_names = info_valid.ch_names
    except Exception:
        info_valid = None
        valid_ch_names = None

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey="row")

    # Top row: time importance
    for j, band_name in enumerate(band_order):
        ax = axes[0, j]
        sub_t = (
            band_time_lda[band_time_lda["band"] == band_name]
            .sort_values("time_s")
            .reset_index(drop=True)
        )

        ax.plot(sub_t["time_s"], sub_t["importance_mean"], linewidth=2)
        ax.axvline(0.0, linestyle="--", alpha=0.7)
        ax.axvspan(late_tmin, late_tmax, alpha=0.15)
        ax.set_title(f"{band_name.capitalize()} time importance")
        ax.set_xlabel("Time (s)")
        ax.grid(True, alpha=0.3)

    axes[0, 0].set_ylabel("Mean normalized importance")

    # Bottom row: topography or fallback bars
    for j, band_name in enumerate(band_order):
        ax = axes[1, j]

        sub_e = (
            band_elec_lda[band_elec_lda["band"] == band_name]
            .copy()
            .groupby("channel", as_index=False)[["importance_mean"]]
            .mean()
        )

        if info_valid is not None and valid_ch_names is not None:
            topo_df = (
                sub_e.set_index("channel")
                .reindex(valid_ch_names)
                .reset_index()
            )

            values_valid = topo_df["importance_mean"].to_numpy(dtype=float)
            values_valid = np.nan_to_num(values_valid, nan=0.0)

            try:
                mne.viz.plot_topomap(
                    values_valid,
                    info_valid,
                    axes=ax,
                    show=False,
                    contours=6,
                    cmap="viridis",
                )
                ax.set_title(f"{band_name.capitalize()} topography")
            except Exception:
                top_df = (
                    sub_e.sort_values("importance_mean", ascending=False)
                    .head(10)
                )
                ax.barh(top_df["channel"][::-1], top_df["importance_mean"][::-1])
                ax.set_title(f"{band_name.capitalize()} top electrodes")
        else:
            top_df = (
                sub_e.sort_values("importance_mean", ascending=False)
                .head(10)
            )
            ax.barh(top_df["channel"][::-1], top_df["importance_mean"][::-1])
            ax.set_title(f"{band_name.capitalize()} top electrodes")

    fig.suptitle(
        f"{MODEL_NAME_MAP.get(PRIMARY_MODEL, PRIMARY_MODEL.upper())} bandpower feature importance: "
        f"time and topography\n{cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
        y=1.02,
    )
    fig.tight_layout()
    return fig

In [ ]:
fig_7_2 = plot_bandpower_report_figure(
    band_time_lda=band_time_lda,
    band_elec_lda=band_elec_lda,
    ch_names=ch_names,
    late_tmin=P300_TMIN,
    late_tmax=P300_TMAX,
)

save_figure(
    fig_7_2,
    FIG_DIR / f"figure_7_2_{PRIMARY_MODEL}_bandpower_diagnostic.png",
)
plt.show()

## 9. Build bandpower helper tables

We now prepare compact helper tables for Chapter 7 writing.

These will summarize:

- mean importance by band
- relative band ranking
- top gamma electrodes

In [ ]:
band_summary_display = (
    band_lda.copy()
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
band_summary_display["rank"] = np.arange(1, len(band_summary_display) + 1)

gamma_top_display = (
    band_elec_lda[band_elec_lda["band"] == "gamma"][["channel", "importance_mean", "importance_std"]]
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(round_numeric_columns(band_summary_display.copy(), 4))
display(round_numeric_columns(gamma_top_display.head(15).copy(), 4))

In [ ]:
save_dataframe(
    band_summary_display,
    TABLE_DIR / "table_7_2a_band_importance_summary.csv",
)

save_dataframe(
    gamma_top_display,
    TABLE_DIR / "table_7_2b_gamma_top_electrodes.csv",
)

save_latex_table(
    round_numeric_columns(band_summary_display.copy(), 4),
    TABLE_DIR / "table_7_2a_band_importance_summary.tex",
    index=False,
)
save_latex_table(
    round_numeric_columns(gamma_top_display.copy(), 4),
    TABLE_DIR / "table_7_2b_gamma_top_electrodes.tex",
    index=False,
)

print("Saved bandpower helper tables.")

## 10. Build one thesis-facing summary table

This final compact table is useful for writing Chapter 7 prose quickly.

In [ ]:
top_electrodes_list = ", ".join(top_electrodes_p300["channel"].head(5).tolist())
top_gamma_list = ", ".join(gamma_top_display["channel"].head(5).tolist())
top_time_values = top_time_points["time_s"].head(5).round(3).astype(str).tolist()

chapter7_summary = pd.DataFrame([
    {
        "finding": "Broadband coefficient structure",
        "summary": "Distributed post-stimulus spatiotemporal importance rather than a single isolated feature",
    },
    {
        "finding": "Strongest P300-window electrodes",
        "summary": top_electrodes_list,
    },
    {
        "finding": "Top time points by importance",
        "summary": ", ".join(top_time_values),
    },
    {
        "finding": "Dominant band overall",
        "summary": str(band_summary_display.iloc[0]["band"]),
    },
    {
        "finding": "Gamma globally dominant?",
        "summary": "No",
    },
    {
        "finding": "Gamma peripheral concern",
        "summary": f"Some peripheral structure remains; top gamma electrodes include {top_gamma_list}",
    },
])

display(chapter7_summary)

In [ ]:
save_dataframe(
    chapter7_summary,
    TABLE_DIR / "table_7_3_chapter7_summary.csv",
)

save_latex_table(
    chapter7_summary,
    TABLE_DIR / "table_7_3_chapter7_summary.tex",
    index=False,
)

print("Saved final Chapter 7 summary table.")

## 11. Final output check

This cell lists the key files generated by this notebook.

In [ ]:
generated_figs = sorted(FIG_DIR.glob("*.png"))
generated_tables = sorted(TABLE_DIR.glob("*"))

manifest = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg_main.dataset.dataset,
    "task": cfg_main.dataset.task,
    "primary_model": PRIMARY_MODEL,
    "p300_window": [P300_TMIN, P300_TMAX],
    "broadband_signal": cfg_main.signal.slug(),
    "bandpower_signal": cfg_bandpower.signal.slug(),
    "broadband_table_dir": str(BROADBAND_TABLE_DIR),
    "bandpower_table_dir": str(BANDPOWER_TABLE_DIR),
    "artifact_dir": str(OUT_DIR),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "figures_png": [p.name for p in generated_figs],
    "tables": [p.name for p in generated_tables],
}

save_json(
    manifest,
    OUT_DIR / "manifest.json",
)

save_experiment_metadata(
    cfg_main,
    extra={
        "notebook": "09_interpretability_figures_tables.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "artifact_manifest": manifest,
    },
)

print("Generated figures:")
for p in generated_figs:
    print("-", p.name)

print("\nGenerated tables:")
for p in generated_tables:
    print("-", p.name)

print("\nManifest:")
display(manifest)